In [2]:
import pandas as pd
pd.read_csv('../data/diabetic_data.csv').shape

(101766, 50)

In [3]:
import sklearn; 
print(sklearn.__version__)

1.8.0


# 01 — Problem Understanding

## Prediction Problem
**Predict, at the moment of patient discharge, whether a diabetic patient 
will be readmitted within 30 days.**

## Target Definition
Binary classification:
- 1 = readmitted within 30 days (`readmitted == '<30'`)
- 0 = not readmitted within 30 days (`readmitted` is `'>30'` or `'NO'`)

Rationale: 30-day readmissions carry financial penalties under US healthcare 
policy. This is the actionable business target.

## Prediction Timing (Leakage Boundary)
The prediction is made **at discharge**. Only features knowable at or before 
discharge are valid. Any feature describing post-discharge events is leakage 
and must be excluded.

## Grain
Each row = one hospital encounter (visit), identified by `encounter_id`.
NOTE: Patients (`patient_nbr`) can appear in multiple rows. This must be 
handled in the train/test split to avoid the same patient appearing in 
both sets (a form of leakage).

## Class Balance
~11% of encounters are <30 readmissions. Imbalanced classification — 
accuracy will be a misleading metric; will use precision, recall, ROC-AUC.

## Business Use
A risk score at discharge lets care teams target follow-up resources 
(calls, home visits, medication checks) at high-risk patients before 
they leave.

In [4]:
pd.read_csv('../data/diabetic_data.csv').head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO
